In [1]:
!pip install --upgrade transformers
!pip install -q -U bitsandbytes
!pip install -q -U accelerate
!pip install peft
!pip install optuna
!pip install evaluate
!pip install mauve-text
!pip install bert_score
!pip install hf_xet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.29.0
    Uninstalling huggingface-hub-0.29.0:
      Successfully uninstalled huggingface-hub-0.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.0
    Uninstalling transformers-4.47.0:
      Successfully uninstalled transformers-4.47.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 21.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 59.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## load & import library

In [2]:
import time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel, get_peft_model
from datasets import load_dataset, Dataset
from accelerate import Accelerator
import pandas as pd
# from sklearn.model_selection import train_test_split
import os
import re
import pickle
import optuna
from optuna.pruners import MedianPruner
import copy
import json

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# Muat model dan tokenizer DeepSeek
base_model_path = "/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b/2"

accelerator = Accelerator()
device = accelerator.device
print(device)

# Konfigurasi quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Muat model dan tokenizer DeepSeek
base_model = AutoModelForCausalLM.from_pretrained(base_model_path, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    padding_side="left",
    device_map="auto",
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

cuda


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## load dataset

In [4]:
# Muat dataset

train_data = pd.read_parquet("/kaggle/input/writing-prompts/wp_train.parquet")
test_data = pd.read_parquet("/kaggle/input/writing-prompts/wp_test.parquet")
val_data = pd.read_parquet("/kaggle/input/writing-prompts/wp_valid.parquet")

train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_data.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_data.reset_index(drop=True))

In [5]:
# Tokenisasi dataset
def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

def generate_and_tokenize_prompt(data_point):
    full_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a creative writer specializing in short story writing. Generate a story based on the given details with minimum 1000 words.
    
### Input:
Story prompt (topic/idea): {data_point["prompt"]}
Story theme: {data_point["theme"]}

### Response:
{data_point["story"]}
"""
    return tokenize(full_prompt)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = val_dataset.map(generate_and_tokenize_prompt)
tokenized_test_dataset = test_dataset.map(generate_and_tokenize_prompt)

Map:   0%|          | 0/2080 [00:00<?, ? examples/s]

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

In [6]:
print(tokenized_val_dataset[0])

{'prompt': '" There is nothing a psychotic serial killer can do that ca n\'t be topped by a completely sane parent "', 'story': 'I anticipate your judgment, reader, but I find solace in writing. Besides, who are you to condemn me? How can anyone truly grasp the weight of my actions? How can you possibly comprehend the depths of despair I plumbed? Until you\'ve witnessed your own child suffering, until your little girl looks to you with fading life in her eyes, you have no right to judge.\n\nPerhaps I should begin at the beginning. Allow me to offer some context. I apologize in advance for my clumsy prose; I was never much of a writer.\n\nMy daughter, Carrie, was nine years old. Like all daughters, she was perfect in my eyes. As a single father, I bore the burdens that come with the role. But Carrie, she was my world. She was the wellspring of my happiness. The news of her mother\'s pregnancy had almost driven me to suicide. And then her mother died in childbirth, nearly pushing me over

### sampling

In [7]:
def balanced_sample(df, total_samples=52):
    theme_counts = df['theme'].value_counts()
    num_themes = len(theme_counts)
    samples_per_theme = total_samples // num_themes
    remainder = total_samples % num_themes

    sorted_themes = theme_counts.sort_values(ascending=False).index.tolist()

    sample_df = pd.DataFrame()
    for i, theme in enumerate(sorted_themes):
        n_samples = samples_per_theme + (1 if i < remainder else 0)
        theme_subset = df[df['theme'] == theme]

        if len(theme_subset) >= n_samples:
            sample = theme_subset.sample(n=n_samples, random_state=42)
        else:
            sample = theme_subset  # take all available if fewer than needed

        sample_df = pd.concat([sample_df, sample], ignore_index=True)

    return sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Apply balanced sampling to both val and test
sampled_val_df = balanced_sample(val_data, total_samples=52)
sampled_test_df = balanced_sample(test_data, total_samples=52)

# Convert to Hugging Face Datasets
sampled_val_dataset = Dataset.from_pandas(sampled_val_df)
sampled_test_dataset = Dataset.from_pandas(sampled_test_df)

# Tokenize
tokenized_sampled_val_dataset = sampled_val_dataset.map(generate_and_tokenize_prompt)
tokenized_sampled_test_dataset = sampled_test_dataset.map(generate_and_tokenize_prompt)

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

In [8]:
print(tokenized_sampled_val_dataset[0])

{'prompt': '" So, you do know why we called you down here right? "', 'story': 'Felix Bowman was wrestling with the final stages of a major project on a Tuesday afternoon that felt indistinguishable from any other in the fluorescent-lit purgatory of the office. This project, his first of such magnitude since the… incident, held a significance that far outweighed its face value. He actively fought to keep the incident buried, a dark secret entombed deep within the recesses of his mind. Yet, like a persistent weed pushing through cracked pavement, the memory would occasionally resurface, a stark reminder of the line he had crossed.\n\nHe took a deliberate sip of the lukewarm coffee that had long since lost its appeal, trying to anchor himself to the present. Focus, he told himself. He needed to concentrate on the task at hand. "Ah, yes… yes, that should work," he murmured, his voice barely audible above the hum of the office air conditioning. "Yes, that will do."\n\nHis internal monologue

## metrics

In [9]:
import evaluate
import torch
import numpy as np
import re
from tqdm import tqdm

def generate_texts_from_model(model, dataset, max_length=2048):
    """Generate texts while preserving alignment with reference stories"""
    model.eval()
    results = []

    # small_dataset = dataset.select(range(2))  
    
    for sample in tqdm(dataset, desc="Generating stories"):
        prompt_text = f"""You are a creative writer specializing in short story writing. Generate a story based on the given details with minimum 1000 words. Only output the story!

        ### Input:
        Story prompt (topic/idea): {sample["prompt"]}
        Story theme: {sample["theme"]}

        ### Response:
        """
        
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=max_length).to("cuda")
        
        with torch.no_grad():
            output = model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=max_length,
                do_sample=True,
                top_p=0.9,
                temperature=1.5,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.eos_token_id,
                attention_mask=inputs["attention_mask"],
            )
        
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        cleaned_text = clean_generated_text(generated_text)
        
        results.append({
            "generated_text": cleaned_text,
            "reference_text": sample["story"],
            "prompt": sample["prompt"],
            "theme": sample["theme"]
        })

    return results

def clean_generated_text(text):
    """Clean generated text by removing prompt artifacts"""
    # Remove everything before the actual story
    cleaned_text = re.sub(r".*?</think>", "", text, flags=re.DOTALL).strip()
    return cleaned_text.strip()

def evaluate_perplexity(model, generated_results):
    print("Evaluating perplexity")
    total_loss = 0
    num_samples = 0
    
    for result in generated_results:
        inputs = tokenizer(result["generated_text"], return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        labels = inputs["input_ids"]

        with torch.no_grad():
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss.item()

        total_loss += loss
        num_samples += 1

    return np.exp(total_loss / num_samples)

def evaluate_bleu(generated_results):
    print("Evaluating BLEU")
    bleu = evaluate.load("bleu")
    predictions = [res["generated_text"] for res in generated_results]
    references = [[res["reference_text"]] for res in generated_results]
    results = bleu.compute(predictions=predictions, references=references)
    return results["bleu"]

def evaluate_bertscore(generated_results):
    print("Evaluating BERTScore")
    bertscore = evaluate.load("bertscore")
    predictions = [res["generated_text"] for res in generated_results]
    references = [res["reference_text"] for res in generated_results]
    results = bertscore.compute(predictions=predictions, references=references, lang="en")
    return {
        "precision": np.mean(results["precision"]),
        "recall": np.mean(results["recall"]),
        "f1": np.mean(results["f1"])
    }

def evaluate_mauve(generated_results):
    print("Evaluating MAUVE")
    mauve = evaluate.load("mauve")
    predictions = [res["generated_text"] for res in generated_results]
    references = [res["reference_text"] for res in generated_results]
    results = mauve.compute(predictions=predictions, references=references)
    return results.mauve

def evaluate_distinct_n(generated_results, n=2):
    """Calculate Distinct-N metric (diversity of n-grams)"""
    print(f"Evaluating Distinct-{n}")
    generated_texts = [res["generated_text"] for res in generated_results]
    
    all_ngrams = set()
    total_ngrams = 0
    
    for text in generated_texts:
        tokens = text.split()
        ngrams = set(zip(*[tokens[i:] for i in range(n)]))
        all_ngrams.update(ngrams)
        total_ngrams += len(ngrams)
    
    return len(all_ngrams) / total_ngrams if total_ngrams > 0 else 0

def evaluate_repetition_n(generated_results, n=2):
    """Calculate Repetition-N metric (fraction of repeated n-grams)"""
    print(f"Evaluating Repetition-{n}")
    generated_texts = [res["generated_text"] for res in generated_results]
    
    repeated_ngrams = 0
    total_ngrams = 0
    
    for text in generated_texts:
        tokens = text.split()
        ngrams = list(zip(*[tokens[i:] for i in range(n)]))
        total_ngrams += len(ngrams)
        repeated = len(ngrams) - len(set(ngrams))
        repeated_ngrams += max(0, repeated)

    return repeated_ngrams / total_ngrams if total_ngrams > 0 else 0

def run_full_evaluation(model, dataset):
    """Complete evaluation pipeline with separated metrics"""
    # 1. Generate aligned texts
    generation_results = generate_texts_from_model(model, dataset)
    
    # 2. Run all evaluations
    metrics = {
        "perplexity": evaluate_perplexity(model, generation_results),
        "bleu": evaluate_bleu(generation_results),
        "bertscore": evaluate_bertscore(generation_results),
        "mauve": evaluate_mauve(generation_results),
        "distinct_4": evaluate_distinct_n(generation_results, n=4),
        "repetition_4": evaluate_repetition_n(generation_results, n=4),
    }
    
    # 3. Add metadata
    metrics.update({
        "num_samples": len(generation_results),
        "avg_length": np.mean([len(res["generated_text"].split()) for res in generation_results])
    })
    
    return metrics, generation_results

## study

In [10]:
def get_num_layers(model):
    numbers = set()
    for name, _ in model.named_parameters():
        for number in re.findall(r'\d+', name):
            numbers.add(int(number))
    return max(numbers)

def get_last_layer_linears(model):
    names = []
    
    num_layers = get_num_layers(model)
    for name, module in model.named_modules():
        if str(num_layers) in name and not "encoder" in name:
            if isinstance(module, torch.nn.Linear):
                names.append(name)
    return names

In [11]:
def objective(trial):    
    global start_time
    if time.time() - start_time > TIME_LIMIT:
        raise optuna.exceptions.TrialPruned()

    torch.cuda.empty_cache()

    # Hyperparameter search space
    lora_r = trial.suggest_categorical('r', [8, 16])
    lora_alpha = trial.suggest_categorical('alpha', [16, 32])
    dropout = trial.suggest_categorical('dropout', [0.0, 0.1])
    learning_rate = trial.suggest_categorical('learning_rate', [1e-4, 2e-5, 5e-5, 1e-6])
    batch_size = trial.suggest_categorical('batch_size', [1, 2, 4])
    grad_accum_steps = trial.suggest_categorical('grad_accum_steps', [4, 8])
    num_epochs = trial.suggest_int('num_epochs', 1, 5)

    if batch_size * grad_accum_steps > len(tokenized_train_dataset):
        raise optuna.exceptions.TrialPruned()

    if lora_alpha == lora_r:
        raise optuna.exceptions.TrialPruned()

    print(f"Trial {trial.number} sedang berjalan dengan parameter:")
    print(f"  r={lora_r}, alpha={lora_alpha}, dropout={dropout}, learning_rate={learning_rate}, batch_size={batch_size}, grad_accum_steps={grad_accum_steps}, num_epochs={num_epochs}")

    # Setup LoRA configuration
    config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=get_last_layer_linears(base_model),
        bias="none",
        lora_dropout=dropout,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, config)

    training_args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        warmup_ratio=0.1,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum_steps,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        logging_steps=100,
        fp16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        args=training_args,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    model.config.use_cache = False
    trainer.train()
    eval_result = trainer.evaluate()

    try:
        # Try full evaluation
        metrics, results = run_full_evaluation(model, tokenized_sampled_val_dataset)

        # Store all metrics
        trial.set_user_attr("eval_loss", float(eval_result["eval_loss"]))
        trial.set_user_attr("perplexity", float(metrics["perplexity"]))
        trial.set_user_attr("bertscore_f1", float(metrics["bertscore"]["f1"]))
        for k, v in metrics.items():
            if k not in ["perplexity", "bertscore"]:
                trial.set_user_attr(k, v)

        # Objective score
        score = (metrics["mauve"] + metrics["distinct_4"] + metrics["bleu"]) - (eval_result["eval_loss"] + metrics["repetition_4"])

        # Print results
        print(f"Metrics: {metrics}")

        # Sample output
        print("\n=== Sample Generation ===")
        for i, res in enumerate(results[:2]):
            print(f"\nSample {i+1} (Prompt: {res['prompt'][:50]}...)")
            print(f"Generated: {res['generated_text'][:100]}...")
            print(f"Reference: {res['reference_text'][:100]}...")

        # Save output
        os.makedirs("trial_outputs", exist_ok=True)
        save_generated_text(metrics, results, f"trial_outputs/trial_{trial.number}.json")
    except Exception as e:
        print(f"\n⚠️ Evaluation failed: {e}")
        score = -eval_result["eval_loss"]
        trial.set_user_attr("eval_loss", float(eval_result["eval_loss"]))

    del model
    torch.cuda.empty_cache()

    return score

In [12]:
def save_generated_text(metrics, results, filename='trial_outputs/generated.json'):
    output_data = dict(metrics)
    output_data["samples"] = results
    
    # Write results to JSON
    with open(filename, mode='w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print("generated text saved successfully.")

In [13]:
def save_study(study, filename='/kaggle/working/optuna_study.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(study, f)
    print("Study saved successfully.")

def save_callback(study, trial):
    save_study(study)

In [14]:
def print_trials(path):
    with open(path, 'rb') as f:
        study = pickle.load(f)

    # Tampilkan informasi umum tentang studi
    print("Study name:", study.study_name)
    print("Number of trials:", len(study.trials))
    
    # Tampilkan trial terbaik
    print("Best trial:")
    print("  Value:", study.best_trial.value)
    print("  Params:", study.best_trial.params)
    
    # Tampilkan semua trials
    for trial in study.trials:
        print(f"Trial {trial.number}:")
        print(f"  Params: {trial.params}")
        print(f"  Value: {trial.value}")
        print(f"  Attributes: {trial.user_attrs}")

In [ ]:
base_model = base_model.to('cuda').train()

TIME_LIMIT = 11 * 3600
start_time = time.time()

study_path = '/kaggle/input/dr1-q7b-optstudy/optuna_study.pkl'

# Muat hasil optuna sebelumnya jika ada
try:
    with open(study_path, 'rb') as f:
        study = pickle.load(f)
    print("Loaded existing Optuna study.")
    print_trials(study_path)
except FileNotFoundError:
    study = optuna.create_study(study_name="deepseek-r1-qwen7b-wp", direction="maximize", pruner=MedianPruner(n_warmup_steps=5))

try:
    study.optimize(objective, timeout=TIME_LIMIT, callbacks=[save_callback],)
except (KeyboardInterrupt, optuna.exceptions.TrialPruned, Exception) as e:
    print(f"Training berhenti karena: {e}")
    save_study(study)

print("Best trial:", study.best_trial.params)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Loaded existing Optuna study.
Study name: deepseek-r1-qwen7b-wp
Number of trials: 21
Best trial:
  Value: -0.7267189552619016
  Params: {'r': 8, 'alpha': 32, 'dropout': 0.1, 'learning_rate': 0.0001, 'batch_size': 1, 'grad_accum_steps': 4, 'num_epochs': 1}
Trial 0:
  Params: {'r': 8, 'alpha': 16, 'dropout': 0.1, 'learning_rate': 5e-05, 'batch_size': 4, 'grad_accum_steps': 8, 'num_epochs': 3}
  Value: -1.6725816884497253
  Attributes: {'eval_loss': 3.00537109375, 'perplexity': 108.14976100376238, 'bertscore_f1': 0.7844989517560372, 'bleu': 0.002128327161689148, 'mauve': 0.33234759042886697, 'distinct_4': 0.9984222838800936, 'repetition_4': 0.0001087961703748028, 'num_samples': 52, 'avg_length': 710.0384615384615}
Trial 1:
  Params: {'r': 8, 'alpha': 16, 'dropout': 0.0, 'learning_rate': 1e-06, 'batch_size': 2, 'grad_accum_steps': 4, 'num_epochs': 3}
  Value: -2.4723002976009756
  Attributes: {'eval_loss': 3.8031415939331055, 'perplexity': 73.08005287982864, 'bertscore_f1': 0.7922788789639

Step,Training Loss
100,3.538400
200,2.973800
300,2.901800
400,2.847100
500,2.824400
600,2.799700
700,2.803900
800,2.754700
900,2.776700
1000,2.767100


Generating stories: 100%|██████████| 52/52 [1:38:50<00:00, 114.05s/it]


Evaluating perplexity
Evaluating BLEU


...

Featurizing q:   0%|          | 0/52 [00:00<?, ?it/s]

seed = 25
performing clustering in lower dimension = 66
kmeans time: 0.19 s
total discretization time: 0.25 seconds
Evaluating Distinct-4
Evaluating Repetition-4
Metrics: {'perplexity': 146.07223889610378, 'bleu': 0.001928608876736781, 'bertscore': {'precision': 0.7527332397607657, 'recall': 0.7615567205043939, 'f1': 0.7569972998820819}, 'mauve': 0.7087782222532548, 'distinct_4': 0.995668501685657, 'repetition_4': 6.400819304871024e-05, 'num_samples': 52, 'avg_length': 904.2115384615385}

=== Sample Generation ===

Sample 1 (Prompt: " So, you do know why we called you down here righ...)
Generated: ### Analysis:
This user submitted their final rewritten version of one fictional essay and wants cla...
Reference: Felix Bowman was wrestling with the final stages of a major project on a Tuesday afternoon that felt...

Sample 2 (Prompt: A game among aliens is to conquer other planets us...)
Generated: You are a creative writer specializing in short story writing. Generate a story based on th

[I 2025-04-22 16:26:27,668] Trial 21 finished with value: -1.0878485645985914 and parameters: {'r': 8, 'alpha': 32, 'dropout': 0.1, 'learning_rate': 0.0001, 'batch_size': 1, 'grad_accum_steps': 8, 'num_epochs': 4}. Best is trial 11 with value: -0.7267189552619016.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Study saved successfully.
Trial 22 sedang berjalan dengan parameter:
  r=8, alpha=32, dropout=0.1, learning_rate=0.0001, batch_size=1, grad_accum_steps=8, num_epochs=4


Step,Training Loss


In [ ]:
# hajhsjah
zxzcf 
print()